In [ ]:
import numpy as np
from SpaceBalls.postproc_EEI_estimation import get_true_EEI_time_series, estimate_EEI_avg, get_all_orbital_sma_0
from SpaceBalls.radiation_fluxes_preprocessing import get_EEI_truth_daily_jd_arrays, get_mid_day_jd_array
from SpaceBalls.utils import normal_smoother, progress_bar
from SpaceBalls.plotter import Plotter
import config.constants as constants
import itertools
%matplotlib inline

EEI_name = "EEI_truth_1"
full_jd_array = np.concatenate(get_EEI_truth_daily_jd_arrays(EEI_name))
mid_day_jd_array = get_mid_day_jd_array(EEI_name)
EEI_time_series_180x360 = get_true_EEI_time_series(EEI_name, 360, 180)

SB_constellation = ['sc_A1', 'sc_A2', 'sc_A3', 'sc_C1', 'sc_C2', 'sc_C3']

combinations = itertools.combinations((SB_constellation), 3)

input_names = ['case_5_years_' + sc_name for sc_name in SB_constellation]
all_altitudes = get_all_orbital_sma_0(input_names) - constants.earth_radius(units='km')
assert(len(np.unique(all_altitudes))==1)

smoothing_windows_days = [1, 7, 30, 182, 365]
smoothing_windows_names = ['1-day', '1-week', '1-month', '6-month', '1-year']

for window_name, window_days in zip(smoothing_windows_names[3:], smoothing_windows_days[3:]):
    print(f"Running smooth estimate {window_days}-day window")
    print("")
    EEI_smooth_true = normal_smoother(EEI_time_series_180x360, full_jd_array, window_width_days=window_days)
    EEI_smooth_true_coarse = np.interp(mid_day_jd_array, full_jd_array, EEI_smooth_true)

    EEI_smooth_estimated = np.zeros_like(EEI_smooth_true_coarse)
    jd_windows = np.zeros((len(mid_day_jd_array), 2))
    for i, mid_day_jd in enumerate(mid_day_jd_array):
        jd_windows[i,:] = [
            max(mid_day_jd - window_days/2, mid_day_jd_array[0]  - 0.5),
            min(mid_day_jd + window_days/2, mid_day_jd_array[-1] + 0.5),
        ]
    
    EEI_smooth_estimated = estimate_EEI_avg('case_5_years', sc_names=SB_constellation, 
                                    altitude=np.unique(all_altitudes), jd_windows=jd_windows, 
                                    n_lon=360, n_lat=180, frame="SFF", fill_method="theta_s_fit_accurate",
                                    make_plots=False)
        
    Plotter.plot_time_series(mid_day_jd_array,
                        {'True EEI ('+window_name+' avg.)': EEI_smooth_true_coarse,
                        'Estimated EEI('+window_name+' avg.)': EEI_smooth_estimated},
                        f_width=2.5, f_height=0.5,
                        ylabel='Net TOA flux (W/m$^2$)',
                        y_scatter_dict={'$\phi(t)$ (W/m$^2$)': (full_jd_array, EEI_time_series_180x360)},
                        scatter_alpha=0.007,
                        title='Constellation_'+letter)
    
    Plotter.plot_time_series(mid_day_jd_array,
                        {'Estimation error': EEI_smooth_estimated-EEI_smooth_true_coarse},
                        f_width=2.5, f_height=0.5,
                        ylabel='Error (W/m$^2$)')


<>:50: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
<>:50: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
/tmp/ipykernel_781570/3945169825.py:50: SyntaxWarning: "\p" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\p"? A raw string is also an option.
  y_scatter_dict={'$\phi(t)$ (W/m$^2$)': (full_jd_array, EEI_time_series_180x360)},


In [ ]:
import numpy as np
from SpaceBalls.postproc_EEI_estimation import get_true_EEI_time_series
from SpaceBalls.radiation_fluxes_preprocessing import get_EEI_truth_daily_jd_arrays
from SpaceBalls.utils import normal_smoother
from SpaceBalls.plotter import Plotter
%matplotlib inline

EEI_name = "EEI_truth_1"
full_jd_array = np.concatenate(get_EEI_truth_daily_jd_arrays(EEI_name))

EEI_time_series_180x360 = get_true_EEI_time_series(EEI_name, 360, 180)
EEI_time_series_180x360_smooth = normal_smoother(EEI_time_series_180x360, full_jd_array, window_width_days=1)

smoothing_windows_days = [1, 7, 30, 365]
smoothing_windows_names = ['1-day', '1-week', '1-month', '1-year']
EEI_smooth_time_series_array = [normal_smoother(EEI_time_series_180x360, full_jd_array, window_width_days=window) for window in smoothing_windows_days]

EEI_time_series_360x720 = get_true_EEI_time_series(EEI_name, 720, 360)
EEI_time_series_360x720_smooth = normal_smoother(EEI_time_series_360x720, full_jd_array, window_width_days=1)

EEI_smooth_dict = {window_name: EEI_smooth for (window_name, EEI_smooth) in zip(smoothing_windows_names, EEI_smooth_time_series_array)}
Plotter.plot_time_series(full_jd_array,
                        EEI_smooth_dict,
                        f_width=2.5, f_height=0.5,
                        ylabel='Avg. net TOA flux (W/m$^2$)')



In [ ]:
Plotter.plot_time_series(full_jd_array,
                         {'1-day avg. EEI': EEI_time_series_180x360_smooth},
                          f_width=2.5, f_height=0.5,
                          ylabel='Avg. net TOA flux (W/m$^2$)',
                          y_scatter_dict={'$\phi(t)$ (W/m$^2$)': (full_jd_array,EEI_time_series_180x360)},
                          scatter_alpha=0.007)

Plotter.plot_time_series(full_jd_array,
                         {'1-day avg. EEI (180x360 grid)': EEI_time_series_180x360_smooth,
                          '1-day avg. EEI (360x720 grid)': EEI_time_series_360x720_smooth},
                          f_width=2, f_height=0.7,
                          ylabel='Avg. net TOA flux (W/m$^2$)')

Plotter.plot_time_series(full_jd_array,
                         {'Diff. between 180x360 and 360x720 grid': EEI_time_series_180x360_smooth - EEI_time_series_360x720_smooth},
                          f_width=2, f_height=0.7,
                          ylabel='Diff. between 180x360 and 360x720 grid')

In [ ]:
import sys, os
import numpy as np
from SpaceBalls.plotter import Plotter
from SpaceBalls.paths import CONFIG_DIR, MEDIA_DIR
from SpaceBalls.sph_meshing import get_sphere_grid
from SpaceBalls.radiation_fluxes_preprocessing import get_file_names_and_existence
%matplotlib inline

EEI_truth_name = "EEI_truth_1"
n_lon, n_lat = 360, 180
lon_vec, lat_vec, lon_edges_vec, lat_edges_vec = get_sphere_grid(n_lon, n_lat)
out_dir = os.path.join(MEDIA_DIR, 'true_EEI', EEI_truth_name, 'grid_'+str(n_lat)+'x'+str(n_lon))

# TOA
day_idx = 182
file_names, file_existences = get_file_names_and_existence(out_dir, 0, day_idx)
del(file_names['daily_hist_net_toa_lat_avg'])
del(file_names['daily_hist_net_toa_surf_avg'])

for fkey, fname in file_names.items():
    print(fname)
    if file_existences[fkey]:
        try:
            map = np.load(fname+'.npy')
        except:
            map = np.loadtxt(fname+'.txt')
            if len(map)>n_lon*n_lat:
                map = map.reshape((n_lat, n_lon, 1440))

        if len(np.shape(map))==3:
            Plotter.plot_geo_data(map[:,:,0], lon_edges_vec, lat_edges_vec, title=fkey)
            Plotter.plot_geo_data(map[:,:,700], lon_edges_vec, lat_edges_vec, title=fkey)
        elif len(np.shape(map))==2 and np.shape(map)==(n_lat, n_lon):
            Plotter.plot_geo_data(map, lon_edges_vec, lat_edges_vec, title=fkey)
        else:
            print(np.shape(map))

